In [0]:
# Read the Silver tables that contain location data
df_silver_members = spark.read.table("he_prod_incen_analytics_dbw_01.silver.dim_members")
df_silver_providers = spark.read.table("he_prod_incen_analytics_dbw_01.silver.dim_providers")
df_silver_facilities = spark.read.table("he_prod_incen_analytics_dbw_01.silver.dim_facilities")

In [0]:
from pyspark.sql.functions import col, upper, trim, row_number
from pyspark.sql.window import Window

# 1. Select only the geography columns from each Silver table
df_geo_mem = df_silver_members.select("state", "city", "pincode")
df_geo_prov = df_silver_providers.select("state", "city", "pincode")
df_geo_fac = df_silver_facilities.select("state", "city", "pincode")

# 2. Union them all together to get the enterprise-wide geography footprint
df_geo_all = df_geo_mem.unionByName(df_geo_prov).unionByName(df_geo_fac)

# 3. Deduplicate (Keep only unique State/City/Pincode combinations) and drop nulls
df_geo_distinct = df_geo_all.dropDuplicates(["state", "city", "pincode"]).na.drop(subset=["state", "city", "pincode"])

# 4. Generate Surrogate Key: AUTO_INCREMENT
window_spec = Window.orderBy("state", "city", "pincode")
df_gold_geography = df_geo_distinct.withColumn("geography_sk", row_number().over(window_spec))

# 5. Final Selection
df_gold_geography = df_gold_geography.select(
    "geography_sk",
    "state",
    "city",
    "pincode"
)

display(df_gold_geography.limit(10))

In [0]:
gold_table_fqn = "he_prod_incen_analytics_dbw_01.gold.dim_geography"

df_gold_geography.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print("✅ Successfully wrote Derived Dim_Geography to Gold layer.")